# 📘 Semaine 14 — Révision générale & Examen blanc

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 synthèse + 1h30 examen blanc + 1h30 correction/Q&R)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs de la semaine

1. **Synthétiser** l'ensemble du cours (S1 à S13) en une vue cohérente.
2. **Diagnostiquer** les points faibles restants via un examen blanc.
3. **Maîtriser** les formules et registres clés pour l'examen final.
4. **Se préparer** mentalement et méthodologiquement à l'évaluation finale.

---

## 🗺️ Plan de la semaine

| Séquence | Durée | Contenu |
|---|---|---|
| **A — Synthèse générale** | 1h30 | Carte mentale + fiches récapitulatives |
| **B — Examen blanc** | 1h30 | QCM 40 questions + 1 exercice de code |
| **C — Correction & Q&R** | 1h30 | Correction, feedback, révisions libres |

---

## 📊 Barème de l'examen blanc

| Partie | Points | Durée |
|---|---|---|
| QCM (40 questions) | 20 | 45 min |
| Exercice de code | 10 | 30 min |
| Exercice de calcul | 10 | 15 min |
| **Total** | **/40 → ramené sur /20** | **1h30** |

---

# 🎓 PARTIE A — SYNTHÈSE GÉNÉRALE (1h30)

## 🔹 Carte mentale complète du cours

```
                    STM32F103C6T6
                          │
        ┌─────────────────┼─────────────────┐
        │                 │                 │
   ARCHITECTURE       PÉRIPHÉRIQUES     COMMUNICATION
        │                 │                 │
   ┌────┴────┐       ┌────┴────┐        ┌───┴───┐
   │         │       │         │        │       │
  ARM      Bus     GPIO  TIMER/ADC     UART    I2C/SPI
Cortex-M3  AHB/    EXTI    PWM         USART   SMBus
           APB1/                    
           APB2
```

---

## 🔹 Fiche 1 — Architecture ARM Cortex-M3 (S1)

### 📖 Caractéristiques

| Élément | Valeur |
|---|---|
| Architecture | ARMv7-M (Harvard modifiée) |
| Largeur | 32 bits |
| Pipeline | 3 étages (Fetch/Decode/Execute) |
| Jeu d'instructions | Thumb-2 (16 + 32 bits) |
| Registres généraux | R0–R12 |
| Registres spéciaux | R13=SP, R14=LR, R15=PC, PSR |
| NVIC | 240 IRQ max, 16 niveaux priorité |
| Modes | Thread / Handler |
| Privilèges | Privilégié / Non-privilégié |
| États | Running / Sleep / Deep-Sleep |

---

## 🔹 Fiche 2 — Hardware STM32F103C6T6 (S2)

### 📖 Caractéristiques principales

| Paramètre | Valeur |
|---|---|
| Cœur | Cortex-M3 @ 72 MHz |
| Flash | 32 Ko |
| SRAM | 10 Ko |
| GPIO | 37 (LQFP48) |
| Timers | TIM1 (APB2), TIM2/3 (APB1) |
| ADC | 2 × 12 bits, 10 canaux |
| USART | 2 |
| SPI | 2 |
| I2C | 1 |
| DMA | 7 canaux (DMA1) |
| Alimentation | 2.0–3.6 V |

### 📖 Bus et fréquences

| Bus | Fréquence | Périphériques |
|---|---|---|
| **AHB** | 72 MHz | Flash, SRAM, DMA, GPIO |
| **APB2** | 72 MHz | TIM1, USART1, SPI1, ADC, EXTI, AFIO |
| **APB1** | 36 MHz | TIM2/3, USART2, SPI2, I2C1, USB, CAN |

### 📖 Mappe mémoire

| Adresse | Contenu |
|---|---|
| 0x0800_0000 | Flash (32 Ko) |
| 0x2000_0000 | SRAM (10 Ko) |
| 0x4000_0000 | Périphériques APB1 |
| 0x4001_0000 | Périphériques APB2 |
| 0x4002_0000 | Périphériques AHB |
| 0xE000_0000 | NVIC, SysTick |

### 📖 Horloges

```
HSI (8 MHz) ─┐
             ├──▶ PLL ×9 ──▶ SYSCLK (72 MHz)
HSE (8 MHz) ─┘                │
                              ├── AHB /1 → HCLK = 72 MHz
                              ├── APB1 /2 → PCLK1 = 36 MHz
                              └── APB2 /1 → PCLK2 = 72 MHz
```

### 📖 Formules horloge

```
PLL in    = HSE / PREDIV
SYSCLK    = PLL in × PLLMUL
HCLK      = SYSCLK / AHB_prescaler
PCLK1     = HCLK / APB1_prescaler
PCLK2     = HCLK / APB2_prescaler

TIM sur APB1 : F = PCLK1 × 2  si APB1 prescaler ≠ 1
TIM sur APB2 : F = PCLK2
```

---

## 🔹 Fiche 3 — GPIO (S3)

### 📖 Registres

| Registre | Rôle |
|---|---|
| CRL / CRH | Configuration (mode + CNF) |
| IDR | Lecture entrées |
| ODR | Écriture sorties |
| BSRR | Set/Reset atomique |
| LCKR | Verrouillage |

### 📖 Configurations (4 bits par pin)

| MODE | Sortie | Entrée |
|---|---|---|
| 00 | Entrée | Entrée |
| 01 | Sortie 10 MHz | Réservé |
| 10 | Sortie 2 MHz | Réservé |
| 11 | Sortie 50 MHz | Réservé |

| CNF | Entrée | Sortie |
|---|---|---|
| 00 | Analogique | Push-pull |
| 01 | Flottante | Open-drain |
| 10 | Pull-up/down | AF push-pull |
| 11 | Réservé | AF open-drain |

### 📖 HAL / LL / Registres

```c
HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_SET);
LL_GPIO_SetOutputPin(GPIOC, LL_GPIO_PIN_13);
GPIOC->BSRR = GPIO_BSRR_BS13;
```

---

## 🔹 Fiche 4 — EXTI / NVIC (S4)

### 📖 Lignes EXTI

- 16 lignes (EXTI0 à EXTI15)
- Une seule broche par ligne (configurée via AFIO_EXTICR)
- Vecteurs : EXTI0..4 séparés ; EXTI5–9 et EXTI10–15 partagés

### 📖 Registres EXTI

| Registre | Rôle |
|---|---|
| IMR | Masque |
| RTSR | Front montant |
| FTSR | Front descendant |
| PR | Pending |
| SWIER | Forcer par logiciel |

### 📖 Mapping AFIO_EXTICR

```
AFIO_EXTICR1 : EXTI0..3 (4 bits chacune)
  0000 = PAx, 0001 = PBx, 0010 = PCx

AFIO_EXTICR2 : EXTI4..7
AFIO_EXTICR3 : EXTI8..11
AFIO_EXTICR4 : EXTI12..15
```

### 📖 NVIC

- 16 niveaux de priorité (4 bits)
- Priorité **numériquement plus petite** = plus prioritaire
- Préemption (nested) + sous-priorité
- `HAL_NVIC_SetPriority()`, `HAL_NVIC_EnableIRQ()`

---

## 🔹 Fiche 5 — TIMER (S5)

### 📖 Formules

```
                  F_clk
F_timer = ───────────────────────
           (PSC + 1) × (ARR + 1)

T_timer = 1 / F_timer
```

### 📖 Registres

| Registre | Bits | Rôle |
|---|---|---|
| PSC | 16 | Prescaler |
| CNT | 16/32 | Compteur |
| ARR | 16/32 | Auto-reload |
| SR (UIF) | — | Flag update |

### 📖 Timers du STM32F103C6T6

| Timer | Bus | Bits | Canaux |
|---|---|---|---|
| TIM1 | APB2 | 16 | 4 + complémentaires |
| TIM2 | APB1 | 32 | 4 |
| TIM3 | APB1 | 16 | 4 |

### 📖 HAL

```c
HAL_TIM_Base_Start_IT(&htim2);
void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim);
```

---

## 🔹 Fiche 6 — PWM (S6)

### 📖 Formules

```
F_pwm = F_clk / ((PSC + 1) × (ARR + 1))
Duty  = CCR / (ARR + 1)
CCR   = Duty × (ARR + 1)
V_moy = Duty × V_high
```

### 📖 Modes

| Mode | Comportement |
|---|---|
| PWM Mode 1 | Actif tant que CNT < CCR |
| PWM Mode 2 | Actif tant que CNT ≥ CCR |
| Edge-aligned | Comptage up/down |
| Center-aligned | Up puis down |

### 📖 Applications et fréquences

| Application | Fréquence | Duty typique |
|---|---|---|
| LED (dimming) | 1–10 kHz | 0–100 % |
| Servo | 50 Hz | 5–10 % (1–2 ms) |
| Moteur DC | 10–20 kHz | 0–100 % |
| Buzzer | 2–4 kHz | 50 % |

### 📖 HAL

```c
HAL_TIM_PWM_Start(&htim3, TIM_CHANNEL_1);
__HAL_TIM_SET_COMPARE(&htim3, TIM_CHANNEL_1, ccr);
```

---

## 🔹 Fiche 7 — ADC (S8)

### 📖 Caractéristiques

| Paramètre | Valeur |
|---|---|
| Résolution | 12 bits (4096) |
| Canaux externes | 10 (IN0..IN9) |
| V_ref | VDDA (typ. 3.3 V) |
| F_ADC max | 14 MHz |
| Méthode | SAR |

### 📖 Formules

```
Code = round(V_in / V_ref × 4096)
V_in = Code × V_ref / 4096
LSB  = V_ref / 4096 ≈ 0.806 mV (3.3 V)
```

### 📖 Modes

| Mode | Description |
|---|---|
| Simple | 1 canal, 1 conversion |
| Continu | Boucle |
| Scan | Plusieurs canaux |
| DMA | Transfert automatique |

### 📖 HAL

```c
HAL_ADCEx_Calibration_Start(&hadc1);
HAL_ADC_Start(&hadc1);
HAL_ADC_PollForConversion(&hadc1, 100);
uint16_t val = HAL_ADC_GetValue(&hadc1);
HAL_ADC_Stop(&hadc1);
```

---

## 🔹 Fiche 8 — ADC + DMA (S9)

### 📖 DMA1

| Périphérique | Canal DMA |
|---|---|
| ADC1 | Canal 1 |
| SPI1_RX | Canal 2 |
| SPI1_TX | Canal 3 |
| USART1_TX | Canal 4 |
| USART1_RX | Canal 5 |
| I2C1_RX | Canal 7 |

### 📖 Configuration

- Mode : **Circular** pour acquisition continue
- Data width : **Half Word** (16 bits) pour ADC
- Increment Memory : **Enabled**

### 📖 Callbacks

```c
void HAL_ADC_ConvHalfCpltCallback(ADC_HandleTypeDef *hadc);  // demi-buffer
void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc);      // buffer complet
```

### 📖 Filtrage

| Filtre | Formule |
|---|---|
| Moyenne glissante | y = (x[n]+...+x[n-N+1])/N |
| Médiane | valeur centrale sur N échantillons |
| IIR passe-bas | y[n] = α×x[n] + (1-α)×y[n-1] |
| Hystérésis | 2 seuils (haut/bas) pour éviter rebonds |

---

## 🔹 Fiche 9 — Communication I2C / SPI (S10)

### 📖 I2C

| Élément | Détail |
|---|---|
| Lignes | SDA, SCL (2 fils) |
| Adressage | 7 bits (112 adresses) ou 10 bits |
| Vitesses | 100k / 400k / 1M / 3.4M Hz |
| Multi-maître | Oui (arbitrage) |
| Pull-up | Obligatoires (2.2–10 kΩ) |
| Start | SDA ↓ / SCL haut |
| Stop | SDA ↑ / SCL haut |
| ACK | SDA = 0 après 8ᵉ bit |
| Broches | PB6 (SCL), PB7 (SDA) |

### 📖 SPI

| Élément | Détail |
|---|---|
| Lignes | MOSI, MISO, SCK, CS (4 fils) |
| Full-duplex | Oui |
| Vitesses | jusqu'à 18 MHz (STM32F103) |
| Multi-esclave | Oui (CS séparés) |
| Modes | CPOL/CPHA (4 modes) |
| Broches | PA5 (SCK), PA6 (MISO), PA7 (MOSI) |

---

## 🔹 Fiche 10 — UART / SMBus (S11)

### 📖 UART

| Élément | Détail |
|---|---|
| Lignes | TX, RX (2 fils) + RTS/CTS optionnel |
| Trame | Start + Data (7/8/9) + Parity + Stop |
| Baud | 9600 à 921600 (typ.) |
| Asynchrone | Oui (UART) ; USART peut être synchrone |
| Broches | PA2 (TX), PA3 (RX) pour USART2 |
| BRR (115200) | 0x271 (à 72 MHz) |

### 📖 SMBus

| Élément | Détail |
|---|---|
| Basé sur | I2C |
| Timeout | 35 ms obligatoire |
| PEC | CRC-8 (poly 0x07) |
| Fréquence | 10 kHz à 100 kHz |
| V_IH | 2.1 V fixe |
| Applications | Batterie, gestion système |

---

## 📊 Tableau récapitulatif — Broches clés du STM32F103C6T6

| Broche | Fonctions principales |
|---|---|
| PA0 | TIM2_CH1, EXTI0, ADC_IN0 |
| PA1 | TIM2_CH2, EXTI1, ADC_IN1 |
| PA2 | USART2_TX, TIM2_CH3, ADC_IN2 |
| PA3 | USART2_RX, TIM2_CH4, ADC_IN3 |
| PA4 | SPI1_NSS, ADC_IN4 |
| PA5 | SPI1_SCK, ADC_IN5 |
| PA6 | TIM3_CH1, SPI1_MISO, ADC_IN6 |
| PA7 | TIM3_CH2, SPI1_MOSI, ADC_IN7 |
| PA8 | TIM1_CH1, MCO |
| PA9 | USART1_TX, TIM1_CH2 |
| PA10 | USART1_RX, TIM1_CH3 |
| PA13 | SWDIO (debug) |
| PA14 | SWCLK (debug) |
| PB0 | TIM3_CH3, ADC_IN8 |
| PB1 | TIM3_CH4, ADC_IN9 |
| PB6 | I2C1_SCL |
| PB7 | I2C1_SDA |
| PC13 | LED intégrée (Blue Pill) |

---

## 🐍 Simulateur récapitulatif (10 min)

Utilise cette cellule pour vérifier tes calculs sur n'importe quelle config.

In [ ]:
# ============================================================
# Simulateur récapitulatif — Toutes les formules du cours
# ============================================================

F_CLK = 72_000_000   # 72 MHz (SYSCLK / horloge timer)
V_REF = 3.3          # V_ref ADC
BITS_ADC = 12

# --- Timer / PWM ---
def timer_freq(psc, arr):
    return F_CLK / ((psc + 1) * (arr + 1))

def timer_config(f_cible):
    """Retourne PSC, ARR avec PSC+1=72."""
    total = F_CLK / f_cible
    arr = int(total / 72) - 1
    return (71, arr)

def pwm_ccr(duty, arr):
    return int(duty * (arr + 1))

# --- ADC ---
def code_vers_tension(code):
    return code * V_REF / (2 ** BITS_ADC)

def tension_vers_code(v):
    return round(v / V_REF * (2 ** BITS_ADC))

# --- Horloge ---
def clock_tree(hse_mhz, pll_mul, ahb_div=1, apb1_div=2, apb2_div=1):
    sysclk = hse_mhz * pll_mul
    hclk   = sysclk / ahb_div
    pclk1  = hclk / apb1_div
    pclk2  = hclk / apb2_div
    tim_apb1 = pclk1 * (2 if apb1_div != 1 else 1)
    tim_apb2 = pclk2 * (2 if apb2_div != 1 else 1)
    return sysclk, hclk, pclk1, pclk2, tim_apb1, tim_apb2

# --- UART ---
def usart_brr(baud, f_clk=F_CLK):
    usartdiv = f_clk / (16 * baud)
    mantisse = int(usartdiv)
    fraction = round((usartdiv - mantisse) * 16)
    return (mantisse << 4) | (fraction & 0xF)

# ============ DÉMONSTRATION ============
print("═" * 60)
print("🎯 SIMULATEUR RÉCAPITULATIF — STM32F103C6T6")
print("═" * 60)

print("\n⏱️  TIMER :")
for freq in [1, 2, 10, 100, 1000, 10000]:
    psc, arr = timer_config(freq)
    f = timer_freq(psc, arr)
    print(f"  {freq:>5} Hz cible → PSC={psc}, ARR={arr}, F_reelle={f:.4f} Hz")

print("\n📊 PWM :")
psc, arr = timer_config(1000)
for duty in [0, 0.25, 0.5, 0.75, 1.0]:
    ccr = pwm_ccr(duty, arr)
    print(f"  Duty {int(duty*100):>3}% → CCR = {ccr}")

print("\n🔢 ADC (12 bits, 3.3 V) :")
for v in [0, 0.5, 1.0, 1.65, 2.0, 2.5, 3.0, 3.3]:
    c = tension_vers_code(v)
    print(f"  {v:>4.2f} V → Code = {c:>4}  (0x{c:03X})")

print("\n🕐 Horloge :")
s, h, p1, p2, t1, t2 = clock_tree(8, 9)
print(f"  HSE 8 MHz × PLL9 : SYSCLK={s} MHz, HCLK={h} MHz")
print(f"  PCLK1={p1} MHz, PCLK2={p2} MHz")
print(f"  TIM1 (APB2)={t2} MHz, TIM2/3 (APB1)={t1} MHz")

print("\n📡 UART :")
for baud in [9600, 19200, 38400, 57600, 115200, 230400]:
    brr = usart_brr(baud)
    print(f"  Baud {baud:>7} → BRR = 0x{brr:04X}")

---

# 🧪 PARTIE B — EXAMEN BLANC (1h30)

## 📋 Consignes

- **Durée : 1h30**
- **Individuel**, sans documents
- **Calculatrice** autorisée (non programmable)
- Répondre à **toutes** les questions

---

## 📝 PARTIE 1 — QCM (20 points — 45 min)

### Thème A — Architecture (Q1 à Q5)

**Q1.** Le pipeline du Cortex-M3 a :  
A. 2 étages  
B. 3 étages  
C. 5 étages  
D. 7 étages

**Q2.** Le registre **R14** est :  
A. SP  
B. LR  
C. PC  
D. PSR

**Q3.** Le mode **Handler** est utilisé :  
A. Pour le programme principal  
B. Pour une ISR  
C. Pour le debug  
D. Pour le boot

**Q4.** NVIC =  
A. Nested Vectored Interrupt Controller  
B. New Vector Interrupt Counter  
C. Native Vector Interface Circuit  
D. Non-Volatile Interrupt Cache

**Q5.** Combien de niveaux de priorité NVIC ?  
A. 4  
B. 8  
C. 16  
D. 256

### Thème B — Hardware STM32 (Q6 à Q10)

**Q6.** Fréquence max du STM32F103C6T6 :  
A. 24 MHz  
B. 48 MHz  
C. 72 MHz  
D. 100 MHz

**Q7.** Flash / SRAM du STM32F103C6T6 :  
A. 16 Ko / 4 Ko  
B. 32 Ko / 10 Ko  
C. 64 Ko / 20 Ko  
D. 128 Ko / 32 Ko

**Q8.** APB1 tourne au maximum à :  
A. 18 MHz  
B. 36 MHz  
C. 72 MHz  
D. 144 MHz

**Q9.** Pour 72 MHz avec HSE 8 MHz :  
A. PLL ×4  
B. PLL ×6  
C. PLL ×9  
D. PLL ×16

**Q10.** Adresse de base de GPIOC :  
A. 0x4001_0800  
B. 0x4001_0C00  
C. 0x4001_1000  
D. 0x4002_1000

### Thème C — GPIO (Q11 à Q15)

**Q11.** Registre CRL configure les pins :  
A. 0 à 7  
B. 8 à 15  
C. 0 à 15  
D. Toutes

**Q12.** Valeur CNF=10, MODE=11 =  
A. Entrée analogique  
B. Sortie PP 50 MHz  
C. AF PP 50 MHz  
D. AF OD 2 MHz

**Q13.** BSRR permet :  
A. Lecture entrées  
B. Set/Reset atomique  
C. Config  
D. Verrouillage

**Q14.** Open-drain est utilisé pour :  
A. LED  
B. Bus I2C  
C. ADC  
D. Timers

**Q15.** Macro pour activer GPIOA :  
A. `RCC_GPIOA_ENABLE()`  
B. `__HAL_RCC_GPIOA_CLK_ENABLE()`  
C. `HAL_GPIOA_Init()`  
D. `GPIOA_CLOCK_ON()`

### Thème D — EXTI / NVIC (Q16 à Q20)

**Q16.** Nombre de lignes EXTI :  
A. 4  
B. 8  
C. 16  
D. 32

**Q17.** Registre de mapping EXTI :  
A. `EXTI->IMR`  
B. `AFIO->EXTICR`  
C. `NVIC->ISER`  
D. `GPIOA->CRL`

**Q18.** `EXTI->PR` sert à :  
A. Activer EXTI  
B. Acquitter un pending  
C. Lire l'état  
D. Configurer front

**Q19.** Priorité plus élevée :  
A. 0  
B. 1  
C. 15  
D. 255

**Q20.** EXTI15 et EXTI10 partagent :  
A. Le même IMR  
B. Le même vecteur NVIC  
C. La même broche  
D. Le même RTSR

### Thème E — TIMER / PWM (Q21 à Q30)

**Q21.** PSC fait :  
A. 8 bits  
B. 12 bits  
C. 16 bits  
D. 32 bits

**Q22.** Flag de débordement :  
A. CC1IF  
B. UIF  
C. TIF  
D. PR

**Q23.** TIM2 est un compteur :  
A. 8 bits  
B. 16 bits  
C. 24 bits  
D. 32 bits

**Q24.** Horloge TIM2 (APB1=36 MHz, div=2) :  
A. 18 MHz  
B. 36 MHz  
C. 72 MHz  
D. 144 MHz

**Q25.** Duty cycle =  
A. T_off / T  
B. T_on / T  
C. F_pwm / F_clk  
D. ARR / PSC

**Q26.** PWM Mode 1 : actif tant que  
A. CNT < CCR  
B. CNT ≥ CCR  
C. CNT = 0  
D. CNT = ARR

**Q27.** Fréquence servo standard :  
A. 50 Hz  
B. 500 Hz  
C. 1 kHz  
D. 10 kHz

**Q28.** Moteur DC : éviter sifflement →  
A. F_pwm < 1 kHz  
B. F_pwm ≈ 50 Hz  
C. F_pwm > 20 kHz  
D. F_pwm = 100 Hz

**Q29.** Fonction HAL démarrage PWM :  
A. `HAL_TIM_Base_Start()`  
B. `HAL_TIM_PWM_Start()`  
C. `HAL_TIM_PWM_Init()`  
D. `HAL_GPIO_WritePin()`

**Q30.** Pour 1 kHz PWM, ARR = 999, duty 25% → CCR =  
A. 100  
B. 250  
C. 500  
D. 750

### Thème F — ADC / DMA / Communication (Q31 à Q40)

**Q31.** Résolution ADC STM32F103 :  
A. 8  
B. 10  
C. 12  
D. 16

**Q32.** Nombre de canaux ADC externes :  
A. 4  
B. 8  
C. 10  
D. 16

**Q33.** LSB pour V_ref = 3.3 V, 12 bits :  
A. 0.403 mV  
B. 0.806 mV  
C. 1.612 mV  
D. 3.300 mV

**Q34.** Canal DMA de ADC1 :  
A. Canal 1  
B. Canal 2  
C. Canal 3  
D. Canal 7

**Q35.** Callback fin de buffer DMA :  
A. `HAL_ADC_ConvHalfCpltCallback`  
B. `HAL_ADC_ConvCpltCallback`  
C. `HAL_DMA_CompleteCallback`  
D. `HAL_ADC_ErrorCallback`

**Q36.** Nombre de fils SPI :  
A. 2  
B. 3  
C. 4  
D. 5

**Q37.** Nombre de fils I2C :  
A. 2  
B. 3  
C. 4  
D. 5

**Q38.** UART est un bus :  
A. Synchrone  
B. Asynchrone  
C. Full-duplex uniquement  
D. Multi-maître

**Q39.** SMBus diffère de I2C par :  
A. Le nombre de fils  
B. Le timeout et le PEC  
C. La vitesse  
D. L'adressage

**Q40.** PEC SMBus est un CRC de :  
A. 4 bits  
B. 8 bits  
C. 16 bits  
D. 32 bits

### 🐍 Saisie et correction du QCM

In [ ]:
# ============================================================
# Réponses au QCM — Examen blanc
# Remplis chaque valeur par 'A', 'B', 'C' ou 'D'
# ============================================================

reponses = {
     1: '?',  2: '?',  3: '?',  4: '?',  5: '?',
     6: '?',  7: '?',  8: '?',  9: '?', 10: '?',
    11: '?', 12: '?', 13: '?', 14: '?', 15: '?',
    16: '?', 17: '?', 18: '?', 19: '?', 20: '?',
    21: '?', 22: '?', 23: '?', 24: '?', 25: '?',
    26: '?', 27: '?', 28: '?', 29: '?', 30: '?',
    31: '?', 32: '?', 33: '?', 34: '?', 35: '?',
    36: '?', 37: '?', 38: '?', 39: '?', 40: '?',
}

CORRIGE = {
     1: 'B',  2: 'B',  3: 'B',  4: 'A',  5: 'C',
     6: 'C',  7: 'B',  8: 'B',  9: 'C', 10: 'C',
    11: 'A', 12: 'C', 13: 'B', 14: 'B', 15: 'B',
    16: 'C', 17: 'B', 18: 'B', 19: 'A', 20: 'B',
    21: 'C', 22: 'B', 23: 'D', 24: 'C', 25: 'B',
    26: 'A', 27: 'A', 28: 'C', 29: 'B', 30: 'B',
    31: 'C', 32: 'C', 33: 'B', 34: 'A', 35: 'B',
    36: 'C', 37: 'A', 38: 'B', 39: 'B', 40: 'B',
}

THEMES = {
    "Architecture ARM":  range(1, 6),
    "Hardware STM32":    range(6, 11),
    "GPIO":              range(11, 16),
    "EXTI / NVIC":       range(16, 21),
    "TIMER / PWM":       range(21, 31),
    "ADC / DMA / COM":   range(31, 41),
}

def corriger(reponses):
    score_total = 0
    details = {}
    for theme, plage in THEMES.items():
        bonnes = 0
        total = 0
        erreurs = []
        for q in plage:
            total += 1
            rep = reponses.get(q, '?')
            if rep == CORRIGE[q]:
                bonnes += 1
                score_total += 1
            else:
                erreurs.append((q, rep, CORRIGE[q]))
        details[theme] = {"bonnes": bonnes, "total": total, "erreurs": erreurs}
    return score_total, details

score, details = corriger(reponses)

print("═" * 60)
print(f"📊 SCORE QCM : {score} / 40  →  {score/2:.1f} / 20")
print("═" * 60)
print()
for theme, info in details.items():
    pct = info['bonnes'] / info['total'] * 100
    barre = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"{theme:<20} {info['bonnes']}/{info['total']}  {barre}  {pct:.0f}%")
    if info['erreurs']:
        print(f"   Erreurs : {', '.join(f'Q{q}({r}→{c})' for q, r, c in info['erreurs'][:8])}")
    print()

---

## 📝 PARTIE 2 — Exercice de code (10 points — 30 min)

### 🎯 Énoncé

Écrire le **code C** (avec HAL) pour réaliser :

1. Une **LED clignotante** sur PC13 à 4 Hz via TIM2 (interruption).
2. Un **bouton** sur PA0 (EXTI0, front montant) qui change la fréquence entre 1 Hz et 4 Hz (alternance).
3. Un **PWM** sur PA6 (TIM3_CH1) à 1 kHz avec un duty de 50 %.

### 📝 Indications

- TIM2 : PSC = 7199. ARR initial = 2499 (4 Hz).
- TIM3 : PSC = 71, ARR = 999.
- Utiliser des callbacks HAL.
- Déclarer les variables partagées `volatile`.

### 📄 Votre réponse

```c
// Écrivez votre code ici
```

### ✅ Corrigé — Exercice de code

In [ ]:
/* ============================================================
   CORRIGÉ — Examen blanc — Exercice de code
   ============================================================ */

#include "main.h"

TIM_HandleTypeDef htim2;
TIM_HandleTypeDef htim3;

/* --- Tables et variables --- */
static const uint16_t arr_freq[2] = { 9999, 2499 };   // 1 Hz, 4 Hz
static const uint8_t  freq_hz[2]  = { 1, 4 };

volatile uint8_t  etape_freq = 1;   // 0=1Hz, 1=4Hz (démarre à 4 Hz)
volatile uint8_t  diviseur   = 0;

/* --- Callback TIM2 --- */
void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM2)
    {
        uint8_t cible = freq_hz[etape_freq];
        diviseur = (diviseur + 1) % cible;
        if (diviseur == 0)
            HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13);
    }
}

/* --- Callback EXTI --- */
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin)
{
    if (GPIO_Pin == GPIO_PIN_0)
    {
        etape_freq = (etape_freq + 1) % 2;
        __HAL_TIM_SET_AUTORELOAD(&htim2, arr_freq[etape_freq]);
    }
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_TIM2_Init();
    MX_TIM3_Init();

    HAL_TIM_Base_Start_IT(&htim2);
    HAL_TIM_PWM_Start(&htim3, TIM_CHANNEL_1);

    // PWM duty 50%
    __HAL_TIM_SET_COMPARE(&htim3, TIM_CHANNEL_1, 500);

    while (1)
    {
        // Traitement en dehors des ISR
    }
}

### 📊 Barème détaillé de l'exercice de code (10 pts)

| Critère | Points |
|---|---|
| Inclusion et déclarations correctes | 1 |
| Configuration TIM2 (PSC=7199, ARR initial) | 1 |
| Configuration TIM3 PWM (PSC=71, ARR=999) | 1 |
| `HAL_TIM_Base_Start_IT(&htim2)` appelé | 1 |
| `HAL_TIM_PWM_Start(&htim3, CH1)` appelé | 1 |
| Callback TIM2 (toggle PC13 avec diviseur) | 2 |
| Callback EXTI (changement ARR) | 2 |
| Variables `volatile` | 1 |
| **Total** | **/10** |

---

## 📝 PARTIE 3 — Exercice de calcul (10 points — 15 min)

### 🎯 Énoncé

Pour le STM32F103C6T6 avec F_clk = **72 MHz** :

**1.** (2 pts) Calculer PSC et ARR pour un timer à **500 Hz**.  
**2.** (2 pts) Pour un PWM à **2 kHz**, ARR = 999, donner CCR pour un duty de **75 %**.  
**3.** (2 pts) Servo à **50 Hz**, ARR = 19999. Donner CCR pour une position à **135°** (T_on linéaire entre 1 ms et 2 ms).  
**4.** (2 pts) ADC 12 bits, V_ref = 3.3 V. Code correspondant à **V_in = 1.2 V** ?  
**5.** (2 pts) Baud rate **57600**, F_clk = 72 MHz. Donner la valeur de `USART_BRR`.

### 📄 Votre réponse

1. ...  
2. ...  
3. ...  
4. ...  
5. ...

In [ ]:
# Corrigé Exercice de calcul
F_CLK = 72_000_000

# 1. Timer 500 Hz (PSC+1 = 72)
f_cible = 500
total = F_CLK / f_cible              # = 144 000
psc_m1 = 72
arr_m1 = int(total / psc_m1)         # = 2000
psc = psc_m1 - 1
arr = arr_m1 - 1
print(f"1. Timer 500 Hz : PSC={psc}, ARR={arr}")
print(f"   Vérif : 72e6 / (72 × 2000) = {F_CLK/(psc_m1*arr_m1):.2f} Hz")

# 2. PWM 2 kHz, ARR=999, duty 75%
arr2 = 999
duty = 0.75
ccr2 = int(duty * (arr2 + 1))
print(f"\n2. PWM 2 kHz, ARR=999, duty 75% : CCR = {ccr2}")

# 3. Servo 50 Hz, ARR=19999, angle 135°
arr3 = 19999
angle = 135
t_on_ms = 1.0 + (angle / 180) * 1.0    # 1 ms à 2 ms
ccr3 = int(t_on_ms / 20 * (arr3 + 1))
print(f"\n3. Servo 135° : T_on = {t_on_ms:.3f} ms, CCR = {ccr3}")

# 4. ADC 12 bits, V=1.2 V
v_in = 1.2
v_ref = 3.3
code = round(v_in / v_ref * 4096)
print(f"\n4. ADC V=1.2V : Code = {code}  (0x{code:03X})")

# 5. BRR pour 57600
baud = 57600
usartdiv = F_CLK / (16 * baud)
mantisse = int(usartdiv)
fraction = round((usartdiv - mantisse) * 16)
brr = (mantisse << 4) | (fraction & 0xF)
print(f"\n5. BRR 57600 : Mantisse={mantisse}, Fraction={fraction}, BRR = 0x{brr:04X}")

---

# 🎓 PARTIE C — CORRECTION & Q&R (1h30)

## 📊 Grille globale de l'examen blanc

| Partie | Note | Barème |
|---|---|---|
| QCM (40 q) | /20 | 0.5 pt / question |
| Exercice de code | /10 | Voir barème |
| Exercice de calcul | /10 | 2 pts / question |
| **Total** | **/40** | |
| **Note finale** | **/20** | |

---

## 🎯 Conseils pour l'examen final

### ⏱️ Gestion du temps

1. **Lire** l'ensemble du sujet (2 min).
2. **Commencer** par les questions faciles.
3. **Passer** les questions difficiles et y revenir.
4. **Garder** 10 min pour relire.

### 📖 Formules à mémoriser

```
Timer      : F = F_clk / ((PSC+1) × (ARR+1))
PWM        : Duty = CCR / (ARR+1)
ADC        : Code = V_in / V_ref × 2^n
Horloge    : SYSCLK = HSE × PLLMUL
UART       : BRR = F_clk / (16 × baud)
```

### 📖 Registres à connaître par cœur

| Périphérique | Registres |
|---|---|
| GPIO | CRL, CRH, IDR, ODR, BSRR |
| EXTI | IMR, RTSR, FTSR, PR |
| AFIO | EXTICR[1..4] |
| NVIC | ISER, ICER, IPR |
| TIM | PSC, ARR, CNT, CCR1..4, SR |
| ADC | CR1, CR2, SR, DR, SMPR |
| RCC | CR, CFGR, APBxENR |

### 📖 Broches du STM32F103C6T6 à connaître

| Broche | Fonctions |
|---|---|
| PA0 | TIM2_CH1, EXTI0, ADC_IN0 |
| PA1 | TIM2_CH2, EXTI1, ADC_IN1 |
| PA2/PA3 | USART2_TX/RX |
| PA6/PA7 | TIM3_CH1/CH2 |
| PA9/PA10 | USART1_TX/RX |
| PB6/PB7 | I2C1_SCL/SDA |
| PC13 | LED intégrée |
| PA13/PA14 | SWDIO/SWCLK (debug) |

### 📖 Pièges classiques à éviter

| Piège | Solution |
|---|---|
| Oublier `volatile` | Toujours déclarer les variables partagées avec ISR |
| Oublier `HAL_xxx_Start()` | Vérifier les appels de démarrage |
| Confondre PSC/ARR | PSC divise, ARR compte |
| Oublier calibration ADC | `HAL_ADCEx_Calibration_Start()` |
| Confondre front montant/descendant | RTSR=montant, FTSR=descendant |
| Utiliser `HAL_Delay` dans ISR | Interdit (bloque) |
| Oublier NVIC enable | `HAL_NVIC_EnableIRQ()` |

---

## 📇 Fiches de synthèse ultra-courtes

### 🗂️ Fiche mémo 1 — Les 4 formules essentielles

```
1. Timer : F = F_clk / ((PSC+1) × (ARR+1))
2. PWM   : CCR = Duty × (ARR+1)
3. ADC   : Code = V_in / V_ref × 4096
4. UART  : BRR = F_clk / (16 × baud)
```

### 🗂️ Fiche mémo 2 — Les 4 chaînes clés

```
1. GPIO : Pin → CRL/CRH → ODR/BSRR → Sortie
2. EXTI : Pin → AFIO_EXTICR → EXTI → NVIC → ISR
3. TIMER: F_clk → PSC → CNT → ARR → UIF → ISR
4. ADC+DMA: TIM → ADC → DMA → SRAM → CPU
```

### 🗂️ Fiche mémo 3 — Les 4 bus en 4 lignes

```
UART  : 2 fils, asynchrone, point à point, 115200 bds
I2C   : 2 fils, synchrone, multi-maître, 100k-1M, ACK
SPI   : 4 fils, synchrone, full-duplex, 18 MHz, CPOL/CPHA
SMBus : variante I2C + timeout 35 ms + PEC (CRC-8)
```

---

## 🧠 Auto-évaluation finale

### Sur l'ensemble du cours (S1–S14)

Coche ce que tu maîtrises **sans aucune hésitation** :

- [ ] Architecture ARM Cortex-M3 (pipeline, registres, modes)
- [ ] NVIC : priorité, préemption, gestion des IRQ
- [ ] Hardware STM32 : bus, mappe mémoire, horloges
- [ ] Configuration RCC (HSE + PLL → 72 MHz)
- [ ] GPIO : CRL/CRH, IDR/ODR/BSRR (HAL/LL/registres)
- [ ] EXTI : mapping, fronts, pending, ISR
- [ ] TIMER : PSC, ARR, CNT, calculs de fréquence
- [ ] PWM : Duty, CCR, servo, LED, moteur
- [ ] ADC : 12 bits, SAR, conversion, calibration
- [ ] ADC + DMA + trigger timer
- [ ] Filtrage : moyenne, médiane, IIR, hystérésis
- [ ] I2C : adressage, ACK/NACK, START/STOP
- [ ] SPI : 4 fils, CPOL/CPHA, full-duplex
- [ ] UART : trame, BRR, HAL
- [ ] SMBus : timeout, PEC
- [ ] Écriture de code HAL propre (callbacks, volatile)
- [ ] Utilisation de CubeMX (Pinout, Clock, NVIC)
- [ ] Débogage (oscilloscope, terminal série, ST-Link)

### 📊 Mon score : ___ / 18

| Score | Interprétation |
|---|---|
| 16–18 | ✅ Prêt pour l'examen final |
| 12–15 | ⚠️ Revoir 2-3 chapitres ciblés |
| 8–11 | 🔁 Révisions approfondies nécessaires |
| < 8 | ❌ Reprendre le cours à partir de S3 |

---

## 🎯 Plan d'action post-examen blanc

### Si score QCM < 30/40
- **Priorité 1** : réviser les thèmes < 60 %
- **Priorité 2** : refaire les QCM des notebooks concernés
- **Priorité 3** : s'entraîner sur 20 calculs supplémentaires

### Si score code < 6/10
- Réécrire 5 fois de mémoire le squelette d'un projet CubeIDE
- Réviser les callbacks HAL (TIM, EXTI, ADC, DMA)
- S'entraîner à la config CubeMX (minuteur de 10 min par périphérique)

### Si score calculs < 6/10
- Faire 50 calculs PSC/ARR/CCR à la main
- Vérifier avec le simulateur Python
- Mémoriser les formules clés

---

## 📅 Derniers jours avant l'examen

| Jour | Action |
|---|---|
| J-7 | Révision architecture + hardware |
| J-6 | Révision GPIO + EXTI + NVIC |
| J-5 | Révision TIMER + PWM |
| J-4 | Révision ADC + DMA |
| J-3 | Révision Communication (UART/I2C/SPI/SMBus) |
| J-2 | Refaire un examen blanc complet |
| J-1 | Relecture des fiches mémo + reposer le cerveau |
| **J** | **Examen final** |

---

## 📊 Bilan de la semaine 14

| Item | Statut |
|---|---|
| Carte mentale complète lue | ☐ |
| Fiches 1 à 10 revues | ☐ |
| Simulateur Python testé | ☐ |
| QCM examen blanc réalisé | ☐ |
| Exercice code réalisé | ☐ |
| Exercice calculs réalisé | ☐ |
| Auto-évaluation remplie | ☐ |
| Plan d'action défini | ☐ |

### 📈 Note estimée à l'examen blanc : ___ / 20

### 🎯 Objectif personnel : ___ / 20

---

# 📚 RESSOURCES FINALES

### Documents officiels (à relire)
- 📄 **Datasheet STM32F103x6**
- 📄 **RM0008** — Reference Manual
- 📄 **PM0056** — Cortex-M3 Programming Manual
- 📄 **UM1850** — HAL documentation

### Notebooks du cours
- 📓 **S1** — Architecture ARM Cortex-M3
- 📓 **S2** — Hardware STM32F103C6T6
- 📓 **S3** — GPIO (HAL/LL/Registres)
- 📓 **S4** — EXTI + NVIC
- 📓 **S5** — TIMER (base de temps)
- 📓 **S6** — PWM
- 📓 **S7** — QCM (40 questions)
- 📓 **S8** — ADC (principes)
- 📓 **S9** — ADC + DMA
- 📓 **S10** — Communication : I2C et SPI
- 📓 **S11** — Communication : UART/USART et SMBus
- 📓 **S12** — TP noté n°1 (GPIO/EXTI/TIMER/PWM)
- 📓 **S13** — TP noté n°2 (ADC + Communication)
- 📓 **S14** — Révision générale (ce notebook)

### Outils à garder
- **STM32CubeIDE** + **CubeMX**
- **ST-Link V2**
- **Oscilloscope** ou **analyseur logique**
- **Multimètre**
- **Terminal série** (PuTTY, minicom)
- **Carte STM32F103C6T6** (Blue Pill)
- **Composants** : LED, résistances, boutons, potentiomètre, servo

---

## 🎓 Message final

> Vous avez parcouru **14 semaines** d'apprentissage intensif sur les microcontrôleurs STM32.  
> Vous maîtrisez désormais :
> - L'architecture ARM Cortex-M3
> - Le hardware du STM32F103C6T6
> - Les périphériques GPIO, EXTI, TIMER, PWM, ADC, DMA
> - Les bus de communication I2C, SPI, UART, SMBus
> - La programmation HAL et l'accès aux registres
> - La méthode d'analyse et de diagnostic
> 
> **Bonne chance pour l'examen final !** 🚀

---

**Fin du notebook — Semaine 14** ✨  
**Fin du cours Microcontrôleurs STM32F103C6T6** 🎓